In [8]:
import os
import keras

import tensorflow as tf

model_path = "finetuned_model_mix_v2.keras"

try:
    model = keras.models.load_model(model_path, compile=False)
    print("✅ 模型載入成功！")
    
    # 順手找出你的 Grad-CAM 產圖關鍵層
    for layer in reversed(model.layers):
        if 'conv' in layer.name.lower():
            print(f"💡 你的 Grad-CAM 目標卷積層是：{layer.name}")
            break
except Exception as e:
    print(f"❌ 發生錯誤: {e}")

✅ 模型載入成功！
💡 你的 Grad-CAM 目標卷積層是：conv2d_38


In [19]:
import os
import cv2
import numpy as np
import keras
import tensorflow as tf
import matplotlib.pyplot as plt

# ===================== 1. 載入模型 =====================
model = keras.models.load_model(
    "finetuned_model_mix_v2.keras",
    compile=False
)

print("✅ 模型載入成功")

# ===================== 2. 只選 Conv layer（避免炸） =====================
def get_candidate_layers(model):
    return [layer.name for layer in model.layers 
            if isinstance(layer, keras.layers.Conv2D)][-6:]

candidate_layers = get_candidate_layers(model)
print("🎯 候選 layers:", candidate_layers)

# ===================== 3. 前處理 =====================
IMG_SIZE = 224

def preprocess(img_path):
    img = cv2.imread(img_path)
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = img / 255.0
    return np.expand_dims(img, axis=0), img

# ===================== 4. Grad-CAM（防炸版） =====================
def make_gradcam(img_array, model, layer_name):
    try:
        grad_model = keras.models.Model(
            model.inputs,
            [model.get_layer(layer_name).output, model.output]
        )
    except Exception as e:
        print(f"⚠️ 跳過 layer {layer_name}（建 graph 失敗）")
        return None

    with tf.GradientTape() as tape:
        conv_outputs, preds = grad_model(img_array)
        class_idx = tf.argmax(preds[0])
        loss = preds[:, class_idx]

    grads = tape.gradient(loss, conv_outputs)

    if grads is None:
        print(f"⚠️ {layer_name} gradients = None")
        return None

    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    conv_outputs = conv_outputs[0]

    heatmap = tf.reduce_sum(conv_outputs * pooled_grads, axis=-1)
    heatmap = tf.maximum(heatmap, 0)

    if tf.reduce_max(heatmap) == 0:
        print(f"⚠️ {layer_name} heatmap 全 0")
        return None

    heatmap /= tf.reduce_max(heatmap)

    return heatmap.numpy()

# ===================== 5. 自動選最佳 layer =====================
def find_best_layer(img_array, model, candidates):
    best_layer = None
    best_score = -1
    best_heatmap = None

    for layer in candidates:
        heatmap = make_gradcam(img_array, model, layer)
        if heatmap is None:
            continue

        score = np.std(heatmap)

        print(f"✅ {layer} → score: {score:.4f}")

        if score > best_score:
            best_score = score
            best_layer = layer
            best_heatmap = heatmap

    if best_layer is None:
        raise ValueError("❌ 所有 layer 都失敗（請回報我）")

    return best_layer, best_heatmap

# ===================== 6. 疊圖 =====================
def overlay(img, heatmap):
    heatmap = cv2.resize(heatmap, (img.shape[1], img.shape[0]))
    heatmap = np.uint8(255 * heatmap)

    heatmap_color = cv2.applyColorMap(heatmap, cv2.COLORMAP_JET)

    img_bgr = cv2.cvtColor((img * 255).astype(np.uint8), cv2.COLOR_RGB2BGR)

    return cv2.addWeighted(heatmap_color, 0.4, img_bgr, 0.6, 0)

# ===================== 7. 執行 =====================
img_path = "emotion_dataset/Delight/Yo_Delight_25_orig.jpg"

img_array, original_img = preprocess(img_path)

best_layer, heatmap = find_best_layer(img_array, model, candidate_layers)

print(f"\n🏆 最佳 Grad-CAM layer: {best_layer}")

result = overlay(original_img, heatmap)

# ===================== 8. 顯示 =====================
plt.figure(figsize=(10,4))

plt.subplot(1,3,1)
plt.title("Original")
plt.imshow(original_img)
plt.axis("off")

plt.subplot(1,3,2)
plt.title("Heatmap")
plt.imshow(heatmap, cmap='jet')
plt.axis("off")

plt.subplot(1,3,3)
plt.title("Grad-CAM")
plt.imshow(cv2.cvtColor(result, cv2.COLOR_BGR2RGB))
plt.axis("off")

plt.show()

✅ 模型載入成功
🎯 候選 layers: ['conv2d_33', 'conv2d_35', 'conv2d_36', 'conv2d_34', 'conv2d_37', 'conv2d_38']


ValueError: Exception encountered when calling Functional.call().

[1mInput 0 of layer "conv2d_1" is incompatible with the layer: expected axis -1 of input shape to have value 1, but received input with shape (1, 224, 224, 3)[0m

Arguments received by Functional.call():
  • inputs=array([[[[0.07843137, 0.05882353, 0.04705882],
         [0.05490196, 0.05098039, 0.04313725],
         [0.04705882, 0.05098039, 0.05882353],
         ...,
         [0.46666667, 0.40784314, 0.31764706],
         [0.51764706, 0.45882353, 0.36470588],
         [0.5372549 , 0.50588235, 0.49411765]],

        [[0.07058824, 0.0627451 , 0.05490196],
         [0.07058824, 0.06666667, 0.05490196],
         [0.08235294, 0.0745098 , 0.07843137],
         ...,
         [0.4627451 , 0.41568627, 0.32941176],
         [0.50588235, 0.4627451 , 0.41960784],
         [0.54901961, 0.53333333, 0.53333333]],

        [[0.05882353, 0.05882353, 0.06666667],
         [0.0627451 , 0.04705882, 0.05098039],
         [0.09019608, 0.08

In [ ]:
import os
import cv2
import numpy as np
import keras
import tensorflow as tf 
import pandas as pd
from sklearn.metrics import classification_report, accuracy_score

# ===================== 1. 設定區域 =====================
MODEL_PATH = "finetuned_model_mix_v2.keras"
DATASET_DIR = "emotion_dataset" 
OUTPUT_DIR = "results/gradcam_outputs"
EXCEL_PATH = "results/prediction_results.xlsx"

os.makedirs(OUTPUT_DIR, exist_ok=True)

HAAR_DIR = cv2.data.haarcascades
FACE_CASCADE = cv2.CascadeClassifier(os.path.join(HAAR_DIR, "haarcascade_frontalface_default.xml"))
# 你定義的情緒標籤
EMOTION_LABELS = ["挫折", "困惑", "無聊", "喜悅", "投入", "驚訝"]

# 如果你的資料夾名稱是英文，請在這裡對應 (資料夾名稱 : 中文標籤)
# 例如：'Frustration': '挫折'
LABEL_MAP = {
    'Frustration': '挫折',
    'Confusion': '困惑',
    'Boredom': '無聊',
    'Delight': '喜悅',
    'Engagement': '投入',
    'Surprise': '驚訝'
}

model = keras.models.load_model(MODEL_PATH, compile=False)
TARGET_LAYER = "conv2d_38"

# ===================== 2. 影像預處理函數 =====================
def preprocess_for_gradcam(img_path, crop_face=False):
    img_bgr = cv2.imread(img_path)
    if img_bgr is None: return None, None
    
    if crop_face:
        # 使用你原本的臉部偵測邏輯
        frame_gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
        results = FACE_CASCADE.detectMultiScale(frame_gray, 1.1, 5, minSize=(60,60))
        if len(results) > 0:
            x, y, w, h = max(results, key=lambda b: b[2]*b[3])
            roi = img_bgr[y:y+h, x:x+w]
        else:
            return None, None
    else:
        roi = img_bgr # 不裁切，看原圖

    resized = cv2.resize(roi, (224, 224))
    # 這裡配合 v2 模型的預處理：轉灰階 -> 正規化
    gray_resized = cv2.cvtColor(resized, cv2.COLOR_BGR2GRAY)
    face_input = gray_resized.astype(np.float32)[..., None] 
    face_input = np.expand_dims(face_input, axis=0)
    
    return face_input, gray_resized

# ===================== 3. 核心 Grad-CAM 函數 =====================
def get_heatmap(img_input, model, layer_name):
    
    # 先抓 layer（避免 graph 問題）
    target_layer = None
    for layer in model.layers:
        if layer.name == layer_name:
            target_layer = layer
            break

    grad_model = keras.models.Model(
        model.inputs, 
        [target_layer.output, model.layers[-1].input]
    )

    with tf.GradientTape() as tape:
        conv_outputs, logits = grad_model(img_input)
        class_idx = tf.argmax(logits[0])
        loss = logits[:, class_idx]

    grads = tape.gradient(loss, conv_outputs)

    if grads is None:
        return None

    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    
    conv_outputs = conv_outputs[0]

    heatmap = tf.reduce_sum(conv_outputs * pooled_grads, axis=-1)

    heatmap = tf.maximum(heatmap, 0)

    max_val = tf.reduce_max(heatmap)
    if max_val == 0:
        return None

    heatmap /= max_val

    return heatmap.numpy()

# ===================== 3. 執行批次產圖與數據統計 =====================
def run_experiment_with_metrics():
    results_list = []  # 用於存放 Excel 每一列的資料
    y_true = []        # 真實類別
    y_pred = []        # 預測類別

    for folder_name in os.listdir(DATASET_DIR):
        emotion_path = os.path.join(DATASET_DIR, folder_name)
        if not os.path.isdir(emotion_path): continue
        
        # 取得這張圖的真實標籤 (中文)
        actual_label_zh = LABEL_MAP.get(folder_name, folder_name)
        
        save_dir = os.path.join(OUTPUT_DIR, folder_name)
        os.makedirs(save_dir, exist_ok=True)
        
        print(f"📂 正在處理類別：{folder_name} (對應: {actual_label_zh})")
        
        for filename in os.listdir(emotion_path):
            if not filename.lower().endswith(('.png', '.jpg', '.jpeg')): continue
            
            img_path = os.path.join(emotion_path, filename)
            img_input, face_gray = preprocess_for_gradcam(img_path, crop_face=True)
            if img_input is None: continue

            # 1. 執行情緒預測
            preds = model.predict(img_input, verbose=0)
            pred_idx = np.argmax(preds[0])
            pred_label_zh = EMOTION_LABELS[pred_idx]
            confidence = preds[0][pred_idx]

            # 2. 收集統計數據
            y_true.append(actual_label_zh)
            y_pred.append(pred_label_zh)
            
            results_list.append({
                "檔名": filename,
                "真實類別": actual_label_zh,
                "預測類別": pred_label_zh,
                "信心度": round(float(confidence), 4),
                "是否正確": "V" if actual_label_zh == pred_label_zh else "X"
            })

            # 3. 產出 Grad-CAM (Logits 版)
            heatmap = get_heatmap(img_input, model, TARGET_LAYER)
            if heatmap is not None:
                heatmap_res = cv2.resize(heatmap, (224, 224))
                heatmap_color = cv2.applyColorMap(np.uint8(255 * heatmap_res), cv2.COLORMAP_JET)
                face_bgr = cv2.cvtColor(face_gray, cv2.COLOR_GRAY2BGR)
                overlay = cv2.addWeighted(face_bgr, 0.6, heatmap_color, 0.4, 0)
                cv2.imwrite(os.path.join(save_dir, f"cam_{filename}"), overlay)

    # ===================== 4. 產出報表與指標 =====================
    # 儲存 Excel
    df = pd.DataFrame(results_list)
    df.to_excel(EXCEL_PATH, index=False)
    print(f"\n📊 預測細節已匯出至: {EXCEL_PATH}")

    # 計算準確度
    acc = accuracy_score(y_true, y_pred)
    
    # 計算召回率與其他指標
    # 使用 zero_division=0 避免某類別沒樣本時報錯
    report = classification_report(y_true, y_pred, target_names=None, output_dict=True, zero_division=0)
    
    print("\n" + "="*30)
    print(f"🏆 整體準確度 (Accuracy): {acc:.2%}")
    print("-" * 30)
    print(f"{'情緒類別':<10} | {'召回率 (Recall)':<15}")
    print("-" * 30)
    
    for label in EMOTION_LABELS:
        # 檢查該標籤是否有出現在預測或真實資料中
        if label in report:
            recall = report[label]['recall']
            print(f"{label:<12} | {recall:.2%}")
        else:
            print(f"{label:<12} | 0.00% (無樣本)")
    print("="*30)

run_experiment_with_metrics()

📂 正在處理類別：Boredom (對應: 無聊)
📂 正在處理類別：Confusion (對應: 困惑)
📂 正在處理類別：Delight (對應: 喜悅)
📂 正在處理類別：Engagement (對應: 投入)
📂 正在處理類別：Frustration (對應: 挫折)
📂 正在處理類別：Surprise (對應: 驚訝)

📊 預測細節已匯出至: results/prediction_results.xlsx

🏆 整體準確度 (Accuracy): 84.43%
------------------------------
情緒類別       | 召回率 (Recall)   
------------------------------
挫折           | 82.76%
困惑           | 60.87%
無聊           | 62.96%
喜悅           | 100.00%
投入           | 100.00%
驚訝           | 93.33%
